Создадим пример интернет-магазина на Django с полным описанием проекта, приложений и ключевых файлов. Назовем проект **"ShopMaster"**. Это будет интернет-магазин с возможностью добавления товаров, управления категориями, оформления заказов и просмотра товаров.

### **Шаг 1: Создание проекта**

Начнем с создания самого проекта. Откроем терминал и введем следующую команду:

```bash
django-admin startproject ShopMaster
```

Эта команда создаст основную структуру проекта **ShopMaster**.

**Структура проекта**:
```
ShopMaster/
    manage.py
    ShopMaster/
        __init__.py
        settings.py
        urls.py
        asgi.py
        wsgi.py
```

#### **Файлы в корне проекта:**
- `manage.py`: Основная команда для управления проектом — запуск сервера, выполнение миграций, создание суперпользователя и многое другое.
- `settings.py`: Основные настройки проекта (база данных, приложения, статические файлы и т.д.).
- `urls.py`: Файл для управления маршрутами (URL) всего проекта.
- `asgi.py` и `wsgi.py`: Настройки для работы с серверами ASGI и WSGI, необходимыми для деплоя.

---

### **Шаг 2: Создание приложения для управления товарами**

В интернет-магазине ключевая часть — это товары. Создадим первое приложение **catalog**, которое будет отвечать за управление товарами и категориями.

```bash
python manage.py startapp catalog
```

**Структура приложения `catalog`:**
```
catalog/
    migrations/
        __init__.py
    __init__.py
    admin.py
    apps.py
    models.py
    views.py
    urls.py
    templates/
    static/
```

#### **Файлы приложения:**
- `admin.py`: Настройки для отображения моделей в административной панели.
- `models.py`: Определение моделей (товары, категории).
- `views.py`: Логика представлений, которая будет возвращать страницы с товарами.
- `urls.py`: Маршруты для управления страницами товаров.

---

### **Шаг 3: Настройка моделей для каталога товаров**

Теперь настроим модели, чтобы описать товары и категории. Откроем файл `catalog/models.py` и добавим код для создания моделей **Category** и **Product**.

```python
from django.db import models

class Category(models.Model):
    name = models.CharField(max_length=255)
    slug = models.SlugField(unique=True)

    def __str__(self):
        return self.name

class Product(models.Model):
    category = models.ForeignKey(Category, related_name='products', on_delete=models.CASCADE)
    name = models.CharField(max_length=255)
    slug = models.SlugField(unique=True)
    description = models.TextField(blank=True)
    price = models.DecimalField(max_digits=10, decimal_places=2)
    stock = models.PositiveIntegerField()
    available = models.BooleanField(default=True)
    created_at = models.DateTimeField(auto_now_add=True)
    updated_at = models.DateTimeField(auto_now=True)

    def __str__(self):
        return self.name
```

#### **Описание моделей**:
- **Category**: Модель для категорий товаров. У неё есть поля `name` (название категории) и `slug` (уникальная ссылка для категории).
- **Product**: Модель для товаров. Она содержит связь с категорией, а также поля для имени, описания, цены, количества на складе и доступности.

---

### **Шаг 4: Миграции и настройка базы данных**

Теперь нужно применить миграции, чтобы создать таблицы в базе данных. Сначала укажем в `settings.py` приложение `catalog`:

```python
INSTALLED_APPS = [
    ...
    'catalog',
]
```

Затем выполним миграции:

```bash
python manage.py makemigrations
python manage.py migrate
```

Это создаст необходимые таблицы в базе данных для категорий и товаров.

---

### **Шаг 5: Настройка административной панели**

Чтобы управлять товарами и категориями через админку, зарегистрируем модели в `admin.py`:

```python
from django.contrib import admin
from .models import Category, Product

@admin.register(Category)
class CategoryAdmin(admin.ModelAdmin):
    prepopulated_fields = {'slug': ('name',)}

@admin.register(Product)
class ProductAdmin(admin.ModelAdmin):
    list_display = ('name', 'price', 'stock', 'available')
    list_filter = ('available', 'created_at', 'updated_at')
    prepopulated_fields = {'slug': ('name',)}
    search_fields = ('name', 'description')
```

Теперь через административную панель можно добавлять и управлять категориями и товарами.

---

### **Шаг 6: Создание представлений и маршрутов**

Создадим представления для отображения списка товаров и детальной страницы товара. Откроем файл `catalog/views.py`:

```python
from django.shortcuts import render, get_object_or_404
from .models import Category, Product

def product_list(request, category_slug=None):
    category = None
    categories = Category.objects.all()
    products = Product.objects.filter(available=True)
    
    if category_slug:
        category = get_object_or_404(Category, slug=category_slug)
        products = products.filter(category=category)
    
    return render(request, 'catalog/product/list.html', {'category': category, 'categories': categories, 'products': products})

def product_detail(request, id, slug):
    product = get_object_or_404(Product, id=id, slug=slug, available=True)
    return render(request, 'catalog/product/detail.html', {'product': product})
```

#### **Описание представлений**:
- `product_list`: Отображает список товаров, с возможностью фильтрации по категориям.
- `product_detail`: Отображает детальную страницу товара.

Теперь добавим маршруты для этих страниц. Откроем файл `catalog/urls.py`:

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('', views.product_list, name='product_list'),
    path('<slug:category_slug>/', views.product_list, name='product_list_by_category'),
    path('<int:id>/<slug:slug>/', views.product_detail, name='product_detail'),
]
```

---

### **Шаг 7: Настройка шаблонов**

Создадим шаблоны для отображения списка товаров и детальной страницы. В папке `templates/catalog/product/` создадим файлы `list.html` и `detail.html`.

**list.html** (список товаров):

```html
{% extends "base.html" %}

{% block content %}
    <h1>Products</h1>
    <ul>
    {% for product in products %}
        <li>
            <a href="{{ product.get_absolute_url }}">{{ product.name }}</a>
            {{ product.price }} $
        </li>
    {% endfor %}
    </ul>
{% endblock %}
```

**detail.html** (детальная страница товара):

```html
{% extends "base.html" %}

{% block content %}
    <h1>{{ product.name }}</h1>
    <p>{{ product.description }}</p>
    <p>Price: {{ product.price }} $</p>
{% endblock %}
```

---

### **Шаг 8: Подключение маршрутов к основному проекту**

Теперь свяжем маршруты приложения с проектом. Откроем файл `ShopMaster/urls.py` и добавим маршруты для нашего каталога:

```python
from django.contrib import admin
from django.urls import path, include

urlpatterns = [
    path('admin/', admin.site.urls),
    path('', include('catalog.urls', namespace='catalog')),
]
```

---

### **Шаг 9: Запуск проекта**

Теперь можно запустить сервер разработки и проверить наш интернет-магазин:

```bash
python manage.py runserver
```

Перейдите в браузере по адресу `http://127.0.0.1:8000/`, чтобы увидеть список товаров, и кликните на любой товар, чтобы просмотреть его детально.

---

### **Заключение**

Мы создали полноценный интернет-магазин на Django с категориями и товарами, управлением через админку, и системой маршрутизации для отображения товаров.